# Translation (PyTorch)

Install the Transformers, Datasets, and Evaluate libraries to run this notebook.

In [14]:
# Local environment — packages already installed.
# !pip install datasets evaluate transformers[sentencepiece]
# !pip install accelerate
# !apt install git-lfs

You will need to setup git, adapt your email and name in the following cell.

In [15]:
# !git config --global user.email "you@example.com"
# !git config --global user.name "Your Name"

You will also need to be logged in to the Hugging Face Hub. Execute the following and enter your credentials.

In [16]:
from huggingface_hub import notebook_login

notebook_login()

In [17]:
from datasets import load_dataset

raw_datasets = load_dataset("Helsinki-NLP/opus_books", "en-fr")
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['id', 'translation'],
        num_rows: 127085
    })
})

In [18]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['id', 'translation'],
        num_rows: 127085
    })
})

In [19]:
split_datasets = raw_datasets["train"].train_test_split(train_size=0.9, seed=20)
split_datasets

DatasetDict({
    train: Dataset({
        features: ['id', 'translation'],
        num_rows: 114376
    })
    test: Dataset({
        features: ['id', 'translation'],
        num_rows: 12709
    })
})

In [20]:
split_datasets["validation"] = split_datasets.pop("test")

In [21]:
split_datasets["train"][1]["translation"]

{'en': 'I thought Diana very provoking, and felt uncomfortably confused; and while I was thus thinking and feeling, St. John bent his head; his Greek face was brought to a level with mine, his eyes questioned my eyes piercingly--he kissed me.',
 'fr': 'Je trouvai Diana un peu hardie, et je me sentais confuse. Cependant Saint-John pencha sa tête, et sa belle figure grecque se trouva à la hauteur de la mienne; ses yeux perçants interrogeaient les miens.'}

In [22]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_checkpoint = "Helsinki-NLP/opus-mt-en-fr"
_trans_tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
_trans_model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

def translator(text):
    inputs = _trans_tokenizer(text, return_tensors="pt")
    outputs = _trans_model.generate(**inputs)
    return [{"translation_text": _trans_tokenizer.decode(outputs[0], skip_special_tokens=True)}]

translator("Default to expanded threads")

/home/ollie/.cache/pypoetry/virtualenvs/llm-course-hugging-face-_giC5u1K-py3.14/lib/python3.14/site-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

[{'translation_text': 'Par défaut pour les threads élargis'}]

In [23]:
split_datasets["train"][172]["translation"]

{'en': 'If you knew it, you are peculiarly situated: very near happiness; yes, within reach of it. The materials are all prepared; there only wants a movement to combine them. Chance laid them somewhat apart; let them be once approached and bliss results."',
 'fr': "Si vous saviez quelle est votre situation! bien près du bonheur, au moment de l'atteindre; les éléments en sont prêts; il ne faut qu'un seul mouvement pour les réunir: le hasard les a éloignés les uns des autres; qu'ils soient rapprochés, et le résultat sera beau."}

In [24]:
translator(
    "Unable to import %1 using the OFX importer plugin. This file is not the correct format."
)

[{'translation_text': "Impossible d'importer %1 en utilisant le plugin d'importateur OFX. Ce fichier n'est pas le bon format."}]

In [25]:
from transformers import AutoTokenizer

model_checkpoint = "Helsinki-NLP/opus-mt-en-fr"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, return_tensors="pt")

In [26]:
en_sentence = split_datasets["train"][1]["translation"]["en"]
fr_sentence = split_datasets["train"][1]["translation"]["fr"]

inputs = tokenizer(en_sentence, text_target=fr_sentence)
inputs

{'input_ids': [47, 2479, 44904, 420, 1057, 4994, 6608, 2, 10, 5283, 34, 673, 8608, 6655, 25864, 50, 10, 791, 47, 69, 2159, 6067, 10, 8361, 2, 1221, 3, 2211, 45, 313, 179, 2326, 50, 179, 7517, 941, 69, 2506, 12, 15, 518, 42, 4429, 2, 179, 5416, 22040, 240, 5416, 30680, 7512, 244, 21, 21, 2808, 19838, 124, 143, 3, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [131, 34359, 263, 180, 17080, 34, 729, 2625, 894, 2, 11, 157, 143, 39980, 38008, 51, 3, 2373, 1262, 21, 23490, 7007, 6667, 146, 3028, 2, 11, 146, 5455, 2182, 16709, 95, 34359, 17, 8, 6073, 5, 8, 30934, 50, 163, 4219, 329, 25796, 9, 38398, 816, 16, 26882, 9, 3, 0]}

In [27]:
wrong_targets = tokenizer(fr_sentence)
print(tokenizer.convert_ids_to_tokens(wrong_targets["input_ids"]))
print(tokenizer.convert_ids_to_tokens(inputs["labels"]))

['▁Je', '▁tro', 'uv', 'ai', '▁Diana', '▁un', '▁p', 'eu', '▁hard', 'ie', ',', '▁et', '▁je', '▁me', '▁sent', 'ais', '▁confuse', '.', '▁C', 'ep', 'endant', '▁Saint', '-', 'John', '▁pen', 'cha', '▁sa', '▁t', 'ête', ',', '▁et', '▁sa', '▁be', 'lle', '▁figure', '▁gr', 'ec', 'que', '▁se', '▁tro', 'uv', 'a', '▁à', '▁la', '▁ha', 'ute', 'ur', '▁de', '▁la', '▁m', 'ienne', ';', '▁se', 's', '▁ye', 'ux', '▁per', 'ç', 'ants', '▁inter', 'rog', 'ea', 'ient', '▁les', '▁m', 'iens', '.', '</s>']
['▁Je', '▁trouva', 'i', '▁D', 'iana', '▁un', '▁peu', '▁hard', 'ie', ',', '▁et', '▁je', '▁me', '▁sentais', '▁confus', 'e', '.', '▁Cependant', '▁Saint', '-', 'John', '▁pen', 'cha', '▁sa', '▁tête', ',', '▁et', '▁sa', '▁belle', '▁figure', '▁grecque', '▁se', '▁trouva', '▁à', '▁la', '▁hauteur', '▁de', '▁la', '▁mienne', ';', '▁ses', '▁yeux', '▁per', 'çant', 's', '▁interroge', 'aient', '▁les', '▁mien', 's', '.', '</s>']


In [28]:
max_length = 128


def preprocess_function(examples):
    inputs = [ex["en"] for ex in examples["translation"]]
    targets = [ex["fr"] for ex in examples["translation"]]
    model_inputs = tokenizer(
        inputs, text_target=targets, max_length=max_length, truncation=True
    )
    return model_inputs

In [29]:
tokenized_datasets = split_datasets.map(
    preprocess_function,
    batched=True,
    remove_columns=split_datasets["train"].column_names,
)

Map:   0%|          | 0/114376 [00:00<?, ? examples/s]

Map:   0%|          | 0/12709 [00:00<?, ? examples/s]

In [30]:
from transformers import AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

In [31]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

In [32]:
batch = data_collator([tokenized_datasets["train"][i] for i in range(1, 3)])
batch.keys()

KeysView({'input_ids': tensor([[   47,  2479, 44904,   420,  1057,  4994,  6608,     2,    10,  5283,
            34,   673,  8608,  6655, 25864,    50,    10,   791,    47,    69,
          2159,  6067,    10,  8361,     2,  1221,     3,  2211,    45,   313,
           179,  2326,    50,   179,  7517,   941,    69,  2506,    12,    15,
           518,    42,  4429,     2,   179,  5416, 22040,   240,  5416, 30680,
          7512,   244,    21,    21,  2808, 19838,   124,   143,     3,     0],
        [ 1625,  1489,    67,   165,    45,   200,  5532,    21, 19826,  4522,
             2,    12,    45,  1515,     2,   549,   240, 27771,     2,    18,
           271,   862,     3,     0, 59513, 59513, 59513, 59513, 59513, 59513,
         59513, 59513, 59513, 59513, 59513, 59513, 59513, 59513, 59513, 59513,
         59513, 59513, 59513, 59513, 59513, 59513, 59513, 59513, 59513, 59513,
         59513, 59513, 59513, 59513, 59513, 59513, 59513, 59513, 59513, 59513]]), 'attention_mask': tensor(

In [33]:
batch["labels"]

tensor([[  131, 34359,   263,   180, 17080,    34,   729,  2625,   894,     2,
            11,   157,   143, 39980, 38008,    51,     3,  2373,  1262,    21,
         23490,  7007,  6667,   146,  3028,     2,    11,   146,  5455,  2182,
         16709,    95, 34359,    17,     8,  6073,     5,     8, 30934,    50,
           163,  4219,   329, 25796,     9, 38398,   816,    16, 26882,     9,
             3,     0],
        [  104,  7468,  3584,    40,  2244,  5022,    21, 19705, 25514,  1763,
             2,  2392,    17,   183,    23,   203,   735,     2,  1759,    14,
             6,  9225,     5,   505, 27090,     3,     0,  -100,  -100,  -100,
          -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
          -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
          -100,  -100]])

In [34]:
batch["decoder_input_ids"]

tensor([[59513,   131, 34359,   263,   180, 17080,    34,   729,  2625,   894,
             2,    11,   157,   143, 39980, 38008,    51,     3,  2373,  1262,
            21, 23490,  7007,  6667,   146,  3028,     2,    11,   146,  5455,
          2182, 16709,    95, 34359,    17,     8,  6073,     5,     8, 30934,
            50,   163,  4219,   329, 25796,     9, 38398,   816,    16, 26882,
             9,     3],
        [59513,   104,  7468,  3584,    40,  2244,  5022,    21, 19705, 25514,
          1763,     2,  2392,    17,   183,    23,   203,   735,     2,  1759,
            14,     6,  9225,     5,   505, 27090,     3,     0, 59513, 59513,
         59513, 59513, 59513, 59513, 59513, 59513, 59513, 59513, 59513, 59513,
         59513, 59513, 59513, 59513, 59513, 59513, 59513, 59513, 59513, 59513,
         59513, 59513]])

In [35]:
for i in range(1, 3):
    print(tokenized_datasets["train"][i]["labels"])

[131, 34359, 263, 180, 17080, 34, 729, 2625, 894, 2, 11, 157, 143, 39980, 38008, 51, 3, 2373, 1262, 21, 23490, 7007, 6667, 146, 3028, 2, 11, 146, 5455, 2182, 16709, 95, 34359, 17, 8, 6073, 5, 8, 30934, 50, 163, 4219, 329, 25796, 9, 38398, 816, 16, 26882, 9, 3, 0]
[104, 7468, 3584, 40, 2244, 5022, 21, 19705, 25514, 1763, 2, 2392, 17, 183, 23, 203, 735, 2, 1759, 14, 6, 9225, 5, 505, 27090, 3, 0]


In [36]:
# !pip install sacrebleu

In [37]:
import evaluate

metric = evaluate.load("sacrebleu")

In [38]:
predictions = [
    "This plugin lets you translate web pages between several languages automatically."
]
references = [
    [
        "This plugin allows you to automatically translate web pages between several languages."
    ]
]
metric.compute(predictions=predictions, references=references)

{'score': 46.750469682990186,
 'counts': [11, 6, 4, 3],
 'totals': [12, 11, 10, 9],
 'precisions': [91.66666666666667,
  54.54545454545455,
  40.0,
  33.333333333333336],
 'bp': 0.9200444146293233,
 'sys_len': 12,
 'ref_len': 13}

In [39]:
predictions = ["This This This This"]
references = [
    [
        "This plugin allows you to automatically translate web pages between several languages."
    ]
]
metric.compute(predictions=predictions, references=references)

{'score': 1.683602693167689,
 'counts': [1, 0, 0, 0],
 'totals': [4, 3, 2, 1],
 'precisions': [25.0, 16.666666666666668, 12.5, 12.5],
 'bp': 0.10539922456186433,
 'sys_len': 4,
 'ref_len': 13}

In [40]:
predictions = ["This plugin"]
references = [
    [
        "This plugin allows you to automatically translate web pages between several languages."
    ]
]
metric.compute(predictions=predictions, references=references)

{'score': 0.0,
 'counts': [2, 1, 0, 0],
 'totals': [2, 1, 0, 0],
 'precisions': [100.0, 100.0, 0.0, 0.0],
 'bp': 0.004086771438464067,
 'sys_len': 2,
 'ref_len': 13}

In [41]:
import numpy as np


def compute_metrics(eval_preds):
    preds, labels = eval_preds
    # In case the model returns more than the prediction logits
    if isinstance(preds, tuple):
        preds = preds[0]

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    # Replace -100s in the labels as we can't decode them
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Some simple post-processing
    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [[label.strip()] for label in decoded_labels]

    result = metric.compute(predictions=decoded_preds, references=decoded_labels)
    return {"bleu": result["score"]}

In [42]:
from huggingface_hub import notebook_login

notebook_login()

In [43]:
from transformers import Seq2SeqTrainingArguments

args = Seq2SeqTrainingArguments(
    f"marian-finetuned-kde4-en-to-fr",
    eval_strategy="no",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=3,
    predict_with_generate=True,
    bf16=True,
    push_to_hub=True,
)

In [44]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model,
    args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

In [45]:
trainer.evaluate(max_length=max_length)

Training Loss,Validation Loss,Step,Bleu
No log,2.307341,0,21.698397


{'eval_loss': 2.3073408603668213, 'eval_bleu': 21.69839693235175}

In [46]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Step,Training Loss
500,2.026988
1000,1.842737
1500,1.785365
2000,1.736767
2500,1.729845
3000,1.692531
3500,1.686058
4000,1.609277
4500,1.587654
5000,1.594362


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=10725, training_loss=1.6215577139332022, metrics={'train_runtime': 679.3653, 'train_samples_per_second': 505.071, 'train_steps_per_second': 15.787, 'total_flos': 9160244984807424.0, 'train_loss': 1.6215577139332022, 'epoch': 3.0})

In [47]:
trainer.evaluate(max_length=max_length)

Training Loss,Validation Loss,Step,Bleu
1.508404,1.495022,10725,29.289792


{'eval_loss': 1.4950217008590698, 'eval_bleu': 29.28979176266255}

In [48]:
trainer.push_to_hub(tags="translation", commit_message="Training complete")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/ulises-c/marian-finetuned-kde4-en-to-fr/commit/c61a01dc80d83908dfe73c7f056d6cd4b17097d6', commit_message='Training complete', commit_description='', oid='c61a01dc80d83908dfe73c7f056d6cd4b17097d6', pr_url=None, repo_url=RepoUrl('https://huggingface.co/ulises-c/marian-finetuned-kde4-en-to-fr', endpoint='https://huggingface.co', repo_type='model', repo_id='ulises-c/marian-finetuned-kde4-en-to-fr'), pr_revision=None, pr_num=None)

In [49]:
from torch.utils.data import DataLoader

tokenized_datasets.set_format("torch")
train_dataloader = DataLoader(
    tokenized_datasets["train"],
    shuffle=True,
    collate_fn=data_collator,
    batch_size=8,
)
eval_dataloader = DataLoader(
    tokenized_datasets["validation"], collate_fn=data_collator, batch_size=8
)

In [50]:
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

In [51]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=2e-5)

In [52]:
from accelerate import Accelerator

accelerator = Accelerator()
model, optimizer, train_dataloader, eval_dataloader = accelerator.prepare(
    model, optimizer, train_dataloader, eval_dataloader
)

In [53]:
from transformers import get_scheduler

num_train_epochs = 3
num_update_steps_per_epoch = len(train_dataloader)
num_training_steps = num_train_epochs * num_update_steps_per_epoch

lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)

In [54]:
from huggingface_hub import HfApi, create_repo

model_name = "marian-finetuned-kde4-en-to-fr-accelerate"
api = HfApi()
repo_name = f"{api.whoami()['name']}/{model_name}"
create_repo(repo_name, exist_ok=True)
repo_name

'ulises-c/marian-finetuned-kde4-en-to-fr-accelerate'

In [55]:
import os

output_dir = "marian-finetuned-kde4-en-to-fr-accelerate"
os.makedirs(output_dir, exist_ok=True)

In [56]:
def postprocess(predictions, labels):
    predictions = predictions.cpu().numpy()
    labels = labels.cpu().numpy()

    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)

    # Replace -100 in the labels as we can't decode them.
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Some simple post-processing
    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [[label.strip()] for label in decoded_labels]
    return decoded_preds, decoded_labels

In [57]:
from tqdm.auto import tqdm
from huggingface_hub import upload_folder
import torch

progress_bar = tqdm(range(num_training_steps))

for epoch in range(num_train_epochs):
    # Training
    model.train()
    for batch in train_dataloader:
        outputs = model(**batch)
        loss = outputs.loss
        accelerator.backward(loss)

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)

    # Evaluation
    model.eval()
    for batch in tqdm(eval_dataloader):
        with torch.no_grad():
            generated_tokens = accelerator.unwrap_model(model).generate(
                batch["input_ids"],
                attention_mask=batch["attention_mask"],
                max_length=128,
            )
        labels = batch["labels"]

        # Necessary to pad predictions and labels for being gathered
        generated_tokens = accelerator.pad_across_processes(
            generated_tokens, dim=1, pad_index=tokenizer.pad_token_id
        )
        labels = accelerator.pad_across_processes(labels, dim=1, pad_index=-100)

        predictions_gathered = accelerator.gather(generated_tokens)
        labels_gathered = accelerator.gather(labels)

        decoded_preds, decoded_labels = postprocess(predictions_gathered, labels_gathered)
        metric.add_batch(predictions=decoded_preds, references=decoded_labels)

    results = metric.compute()
    print(f"epoch {epoch}, BLEU score: {results['score']:.2f}")

    # Save and upload
    accelerator.wait_for_everyone()
    unwrapped_model = accelerator.unwrap_model(model)
    unwrapped_model.save_pretrained(
        output_dir,
        is_main_process=accelerator.is_main_process,
        save_function=accelerator.save,
        state_dict=accelerator.get_state_dict(model),
    )
    if accelerator.is_main_process:
        tokenizer.save_pretrained(output_dir)
        upload_folder(
            folder_path=output_dir,
            repo_id=repo_name,
            commit_message=f"Training in progress epoch {epoch}",
        )

  0%|          | 0/42891 [00:00<?, ?it/s]

/home/ollie/.cache/pypoetry/virtualenvs/llm-course-hugging-face-_giC5u1K-py3.14/lib/python3.14/site-packages/transformers/data/data_collator.py:600: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  batch["labels"] = torch.tensor(batch["labels"], dtype=torch.int64)


  0%|          | 0/1589 [00:00<?, ?it/s]

epoch 0, BLEU score: 28.87


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

  0%|          | 0/1589 [00:00<?, ?it/s]

epoch 2, BLEU score: 30.35


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

In [58]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Replace this with your own checkpoint
model_checkpoint = "huggingface-course/marian-finetuned-kde4-en-to-fr"
_trans_tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
_trans_model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

def translator(text):
    inputs = _trans_tokenizer(text, return_tensors="pt")
    outputs = _trans_model.generate(**inputs)
    return [{"translation_text": _trans_tokenizer.decode(outputs[0], skip_special_tokens=True)}]

translator("Default to expanded threads")

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/778k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

/home/ollie/.cache/pypoetry/virtualenvs/llm-course-hugging-face-_giC5u1K-py3.14/lib/python3.14/site-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


pytorch_model.bin:   0%|          | 0.00/299M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/299M [00:00<?, ?B/s]

[{'translation_text': 'Par défaut, développer les fils de discussion'}]

In [59]:
translator(
    "Unable to import %1 using the OFX importer plugin. This file is not the correct format."
)

[{'translation_text': "Impossible d'importer %1 en utilisant le module externe d'importation OFX. Ce fichier n'est pas le bon format."}]